In [ ]:
import face_recognition
import cv2
import os
import time

# 1. Define the path (relative to the project root)
# Using os.path.join makes it compatible with both Windows and Mac/Linux
output_folder = os.path.join("..", "data", "detectedFaces")

# Create the folder if it doesn't exist
if not os.path.exists(output_folder):
    print("Directory not found")

# 2. Start Webcam
video_capture = cv2.VideoCapture(0)

print("System Active. Press 'q' to stop.")

last_save = 0
save_delay = 2 # Seconds between captures

while True:
    ret, frame = video_capture.read()
    if not ret: break

    # Performance optimization: Detect on smaller image
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # Detect faces
    face_locations = face_recognition.face_locations(rgb_small)

    # Set padding percentage (0.05 means 5%)
    padding = 0.05

    for face_loc in face_locations:
        # 1. Scale back to original size
        top, right, bottom, left = [v * 4 for v in face_loc]
        
        # Calculate width and height of the detected face
        h = bottom - top
        w = right - left

        # 2. Apply Padding
        # We expand the coordinates but keep them within the image boundaries
        pad_h = int(h * padding)
        pad_w = int(w * padding)

        # New coordinates with padding (using max/min to stay inside image frame)
        # Assuming 'frame' is your original image
        img_h, img_w = frame.shape[:2]
        
        p_top = max(0, top - pad_h)
        p_bottom = min(img_h, bottom + pad_h)
        p_left = max(0, left - pad_w)
        p_right = min(img_w, right + pad_w)

        # 3. Save the Clean Padded Crop
        if time.time() - last_save > save_delay:
            face_img = frame[p_top:p_bottom, p_left:p_right]
            
            timestamp = time.strftime('%Y%m%d_%H%M%S')
            fname = f"face_{timestamp}.jpg"
            save_path = os.path.join("..", "data", "detectedFaces", fname)
            
            cv2.imwrite(save_path, face_img)
            print(f"Saved padded face: {fname}")
            last_save = time.time()

        # 4. Draw the box (for your preview only)
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)

    cv2.imshow('Attendance System - Face Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()

System Active. Press 'q' to stop.
Saved padded face: face_20260209_155132.jpg
Saved padded face: face_20260209_155134.jpg
Saved padded face: face_20260209_155136.jpg
